<a href="https://colab.research.google.com/github/noodlejacknetwork/Disaster-Prediction-2025/blob/main/disaster_events_finals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# DISASTER PREDICTION: AUTO-PILOT VERSION
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- LOAD DATA ---
# PASTE YOUR RAW GITHUB URL BETWEEN THE QUOTES BELOW:
RAW_GITHUB_URL = 'https://github.com/noodlejacknetwork/Disaster-Prediction-2025/blob/main/unclean_dataset.csv'

try:
    df = pd.read_csv(RAW_GITHUB_URL)
    print("✅ Success: Data streamed directly from GitHub!")
except Exception as e:
    print(f"❌ Error: Could not reach the file. Check your URL!")
    print(f"Technical Error: {e}")
    raise

# --- 1. PREPROCESSING ---
# Drop IDs if they exist
if 'event_id' in df.columns:
    df = df.drop(columns=['event_id'])

# Handle Dates
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df = df.drop(columns=['date'])

# Log Transform and One-Hot Encoding
df['affected_population'] = np.log1p(df['affected_population'])
df = pd.get_dummies(df, columns=['disaster_type', 'location'], drop_first=True)

# PCA for Total Impact Score (Combining Severity and Population)
pca_features = ['severity_level', 'affected_population', 'infrastructure_damage_index']
x_pca = StandardScaler().fit_transform(df[pca_features])
pca = PCA(n_components=1)
df['total_impact_score'] = pca.fit_transform(x_pca)

# --- 2. MODEL TRAINING (Random Forest) ---
X = df.drop(columns=['is_major_disaster'])
y = df['is_major_disaster']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Using your specific Phase 3 settings: 260 trees, Depth 20
model = RandomForestClassifier(n_estimators=260, max_depth=20, random_state=42)
model.fit(X_train, y_train)

# --- 3. RESULTS & VISUALS ---
y_pred = model.predict(X_test)

print("\n" + "="*30)
print(f"OVERALL ACCURACY: {accuracy_score(y_test, y_pred):.4f}")
print("="*30)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Visual: Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title("Model Performance: Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# Visual: Feature Importance
plt.figure(figsize=(10, 6))
feat_importances = pd.Series(model.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh')
plt.title("Top 10 Factors Predicting Major Disasters")
plt.show()